# Rolling Calc: Static / Dynamic Acceleration Components

## 滚动窗口计算加速度静态与动态分量

本代码对预处理后的加速度 CSV 数据，采用滚动窗口（窗口 2 秒，采样率 25 Hz）计算静态分量（通过低通滤波或滑动平均）和动态分量（原始减静态），并导出俯仰角、横滚角、VeDBA 等衍生特征。这些分量是论文 2.6 节中为传统机器学习模型（如 XGBoost）提取 119 个手工特征的关键步骤（见表 S6），为对比深度学习与经典方法的性能提供特征工程基础。



## Import Modules

In [5]:
# 导入标准库
import os            # os：文件和目录操作（路径拼接、文件存在检查等）
import glob          # glob：文件路径匹配（用于批量查找 CSV 文件）
import polars as pl  # polars：高性能 DataFrame 库（用于处理大规模加速度数据）

import sys
sys.path.append("../")  # 将项目根目录添加到 Python 模块搜索路径
sys.dont_write_bytecode = True
%load_ext autoreload
%autoreload 2

# =============================================================================
# 导入项目自定义模块
#
# feature_extraction：特征提取模块
#   - calc_statical_and_dynamic_components()：计算加速度的静态和动态分量
#   - 对应论文第 2.6 节：为传统机器学习模型提取 119 个手工特征
#
# utils：工具函数模块
#   - setup_plot()：设置绘图样式
#   - start_time_counter()：计时函数
#   - 其他通用工具函数
#
# 对应论文第 2.6 节：
#   - "The inputs of these models were 119 handcrafted features extracted from raw data."
#   - 静态/动态分量的计算是提取 119 个特征的前置步骤
# =============================================================================
from src import feature_extraction
from src import utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Config

In [6]:
# 选择要处理的物种
# species_list = ["omizunagidori", "umineko"]  # 同时处理两个物种（全量运行）
species_list = ["omizunagidori"]               # 仅处理白额鹱（当前使用）
# species_list = ["umineko"]                   # 仅处理黑尾鸥

debug_test_mode = False
# debug_test_mode = True  # 调试模式（注释掉，当前使用正常模式）

# =============================================================================
# 数据根目录路径

# 路径说明：
#   - 当前 notebook 位于 notebooks/ 子目录下
#   - "../data" 表示上级目录的 data 文件夹
#   - 即项目根目录下的 data/ 文件夹
#
# 数据目录结构：
#   dl-wabc/
#   ├── data/
#   │   ├── datasets/
#   │   │   ├── raw-data/          ← 原始数据
#   │   │   ├── preprocessed_data/ ← 预处理后的数据（CSV）
#   │   │   └── logbot_data/       ← 特征提取中间数据
#   │   │       └── feature_extraction/
#   │   │           ├── data_after_rolling_calc/  ← 滚动计算后（当前输出）
#   │   │           └── acc_features/             ← 119 个手工特征
#   │   └── model-output/           ← 模型训练输出
#   ├── configs/                   ← 配置文件
#   ├── notebooks/                 ← Jupyter Notebook（当前位置）
#   └── src/                       ← 源代码
# =============================================================================
data_dir = "../data"

# =============================================================================
# 步骤 4：滚动窗口参数
#
# 对应论文第 2.6 节（特征提取）：
#   - "using a rolling window to compute static and dynamic acceleration components"
#
# SAMPLING_RATE = 25
#   - 加速度数据采样率：25 Hz（每秒 25 个采样点）
#   - 对应论文第 2.1 节：所有数据重采样至 25Hz
#   - 每个采样点之间的时间间隔：1/25 = 0.04 秒（40 毫秒）
#
# ROLLING_WINDOW_SEC = 2
#   - 滚动窗口长度：2 秒
#   - 对应的采样点数量：25 Hz × 2 秒 = 50 个采样点
#   - 与论文中滑动窗口大小（window_size=50）一致
# =============================================================================
SAMPLING_RATE = 25          # 采样率：25 Hz
ROLLING_WINDOW_SEC = 2      # 滚动窗口长度：2 秒

## Run rolling calc per individual

For each species, glob all preprocessed CSVs, then per-animal:
1. Load the CSV with polars.
2. If there is at least one labelled segment (more than 1 unique `label_id`), compute static/dynamic components.
3. Save to `<output_dir>/<species>/<animal_id>.csv` unless `debug_test_mode` is on.

In [7]:
print("--------------------------------")
print("running rolling calc !")
print("--------------------------------")

# 外层循环：遍历物种列表
for idx, species in enumerate(species_list):

    # --- 步骤 1：设置输入输出路径 ---
    #
    # base_dir：数据根目录
    #   - 指向 data/datasets/logbot_data/
    #
    # input_target：输入文件匹配模式
    #   - 例如：data/datasets/preprocessed_data/omizunagidori/*.csv
    #   - 匹配该物种下所有预处理后的 CSV 文件
    #
    # output_dir：输出目录
    #   - 例如：data/datasets/logbot_data/feature_extraction/data_after_rolling_calc/
    #   - 滚动计算后的数据将保存在此目录下
    base_dir = f"{data_dir}/datasets/logbot_data"
    input_target = f"{data_dir}/datasets/preprocessed_data/{species}/*.csv"
    preprocessed_data_path_list = sorted(glob.glob(input_target))
    print("N of individuals:", len(preprocessed_data_path_list))

    output_dir = f"{base_dir}/feature_extraction/data_after_rolling_calc/"

    # --- 步骤 2：测试模式（可选） ---
    # 如果取消注释，仅处理前 4 个个体用于快速测试
    # preprocessed_data_path_list = preprocessed_data_path_list[:4]

    # --- 步骤 3：内层循环：遍历该物种下的每个个体 ---
    for preprocessed_data_path in preprocessed_data_path_list:

        # --- 步骤 3a：启动计时器 ---
        # 记录当前个体的处理时间
        start_time_perf, start_time_process = utils.start_time_counter()

        # --- 步骤 3b：获取个体 ID ---
        # 从文件路径中提取文件名，去掉 .csv 后缀
        # 例如：OM1803.csv → OM1803
        animal_id = os.path.basename(preprocessed_data_path).replace(".csv", "")
        print("-----------------------------------------------------------------")
        print(f"Loading data for: {animal_id}")

        # --- 步骤 3c：使用 polars 读取 CSV ---
        # polars 是高性能 DataFrame 库，适合处理大规模数据
        # 读取的 df 包含列：
        #   - datetime, unixtime, acc_x, acc_y, acc_z, label, label_id
        df = pl.read_csv(preprocessed_data_path)

        # --- 步骤 3d：检查是否有标签数据 ---
        # n_unique_labels = len(df['label_id'].unique())
        #   - 如果所有 label_id 都是 NaN，则 n_unique_labels = 1
        #   - 如果存在有效标签，则 n_unique_labels > 1
        n_unique_labels = len(df['label_id'].unique())
        if n_unique_labels > 1:  # all nan -> 1, one or more labels -> =< 2

            # --- 步骤 3e：计算静态和动态分量 ---
            # 调用 feature_extraction.calc_static_and_dynamic_components()
            #
            # 参数：
            #   - df：输入的加速度 DataFrame
            #   - sampling_rate=25：采样率 25 Hz
            #   - rolling_window_sec=2：滚动窗口 2 秒
            #
            # 计算内容（对应论文第 2.6 节）：
            #   - 静态分量（acc_x_st, acc_y_st, acc_z_st）
            #   - 动态分量（acc_x_dy, acc_y_dy, acc_z_dy）
            #   - 俯仰角（pitch）
            #   - 横滚角（roll）
            #   - VeDBA（矢量动态身体加速度）
            df = feature_extraction.calc_static_and_dynamic_components(
                df,
                sampling_rate=SAMPLING_RATE,
                rolling_window_sec=ROLLING_WINDOW_SEC
            )

            # --- 步骤 3f：转换为 pandas DataFrame 并预览 ---
            # polars → pandas 转换（便于保存为 CSV）
            # 打印前 3 行以验证计算是否正确
            df_labelled = df
            print(f"len(df_labelled): {len(df_labelled)}")
            df_labelled = df_labelled.to_pandas()
            print(df_labelled.head(3))

            # --- 步骤 3g：构建输出路径并保存 ---
            # 输出路径：data/datasets/logbot_data/feature_extraction/data_after_rolling_calc/{species}/{animal_id}.csv
            df_save_dir = f"{output_dir}/{species}"
            os.makedirs(df_save_dir, exist_ok=True)
            df_save_path = f"{df_save_dir}/{animal_id}.csv"

            print(f"df_save_path: {df_save_path}")

            # --- 步骤 3h：根据调试模式决定是否保存 ---
            if debug_test_mode == True:
                print("| debug test mode -> do not save data |")
            else:
                df_labelled.to_csv(df_save_path, index=False)

        else:
            # --- 步骤 3i：无标签数据 ---
            # 如果该个体没有标签数据，跳过滚动计算
            # 例如：OM1802、OM1810、OM1811、OM2206、OM2209
            print("No labelled data")

        # --- 步骤 3j：结束计时并打印耗时 ---
        # 输出该个体的处理时间
        # 便于识别处理速度较慢的个体
        elapsed_time_perf, elapsed_time_process = utils.end_time(
            start_time_perf, start_time_process
        )

# 打印滚动计算完成信息
print("--------------------------------")
print("rolling calc completed !")
print("--------------------------------")

--------------------------------
running rolling calc !
--------------------------------
N of individuals: 33
-----------------------------------------------------------------
Loading data for: OM1802
No labelled data
elapsed_time_perf:  00:00:00
elapsed_time_process:  00:00:00
-----------------------------------------------------------------
Loading data for: OM1803
len(df_labelled): 1008000
                  datetime      unixtime     acc_x     acc_y     acc_z label  \
0  2018-09-08 20:13:05.000  1.536438e+09 -0.300781 -0.726563  1.965332  None   
1  2018-09-08 20:13:05.040  1.536438e+09  0.113770 -0.344238  1.512207  None   
2  2018-09-08 20:13:05.080  1.536438e+09 -0.113281 -0.181641  1.469238  None   

  label_id  acc_x_st  acc_y_st  acc_z_st  pitch  roll  acc_x_dy  acc_y_dy  \
0     None       NaN       NaN       NaN    NaN   NaN       NaN       NaN   
1     None       NaN       NaN       NaN    NaN   NaN       NaN       NaN   
2     None       NaN       NaN       NaN    NaN   Na